# PPO Analyse de l'entrainement automatique

-----------------------------------------

### Import :

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Config du dataset :

In [7]:
# Configuration de l'esthétique des graphiques
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)

# 1. Charger les données
# Remplace le chemin par le nom exact de ton fichier CSV
df = pd.read_csv('../documentation/leaderboard_analytics_ppo.csv')

# 2. Nettoyage
# Si train_convergence_ep contient "Fail", on le remplace par le nombre max d'épisodes (ou NaN)
if 'train_convergence_ep' in df.columns:
    df['train_convergence_ep'] = pd.to_numeric(df['train_convergence_ep'].replace('Fail', np.nan))

print(f"Total des tests : {len(df)} | Tests valides et stables : {len(df_valid)}")
df.head(3)

Total des tests : 324 | Tests valides et stables : 0


,test_id,run_name,is_valid,eval_score_mean,eval_std_dev,margin_diff_pct,train_score_final,train_max_avg_score,train_convergence_ep,training_time_sec,...,eval_seed_677,eval_seed_999,train_landed_pct,train_crash_pct,train_out_of_view_pct,train_truncated_pct,eval_landed_pct,eval_crash_pct,eval_out_of_view_pct,eval_truncated_pct
0,3,test_003_PPO_ep7500_lr0.0003_updt1024,False,235.094267,20.93,18.632477,224.816060,232.00,4515.0,2407.00,...,267.139049,224.320819,34.07,53.95,1.77,10.21,92.6,7.2,0.0,0.2
1,5,test_005_PPO_ep7500_lr0.0003_updt1024,False,221.550310,27.27,27.118740,231.100275,233.90,5870.0,2905.38,...,256.266807,220.073642,18.80,59.23,1.56,20.41,89.0,10.8,0.0,0.2
2,4,test_004_PPO_ep7500_lr0.0003_updt1024,False,233.882355,11.47,10.324662,223.957351,254.27,3789.0,3062.72,...,221.711674,228.074498,43.89,35.37,1.01,19.72,88.4,11.6,0.0,0.0


--------------------------------------------

## Top 5 des meilleurs configuration d'hyper parameters

Voici le top 5 des meilleurs scores on vois des scores tres prometteur mais malheureusement l'ecart type n'est pas dutout satisfaisant nous devons avoir 5.

In [9]:
# Sélection des colonnes les plus intéressantes pour le résumé
cols_to_show = ['run_name', 'eval_score_mean', 'eval_std_dev', 
                'param_learning_rate', 'param_update_timestep', 
                'param_ppo_epochs', 'param_hidden_size', 'eval_landed_pct']

top_models = df.sort_values(by='eval_score_mean', ascending=False).head(5)

print("🌟 TOP 5 DES MEILLEURS MODÈLES 🌟")
display(top_models[cols_to_show])

🌟 TOP 5 DES MEILLEURS MODÈLES 🌟


,run_name,eval_score_mean,eval_std_dev,param_learning_rate,param_update_timestep,param_ppo_epochs,param_hidden_size,eval_landed_pct
167,test_167_PPO_ep10000_lr0.0005_updt2048,272.184603,20.71,0.0005,2048,10,64,99.4
53,test_056_PPO_ep7500_lr0.0005_updt2048,270.131460,21.35,0.0005,2048,10,128,99.2
238,test_238_PPO_ep12500_lr0.0003_updt2048,269.141225,23.03,0.0003,2048,10,128,99.4
271,test_273_PPO_ep12500_lr0.0005_updt2048,268.765696,20.03,0.0005,2048,10,64,98.4
164,test_166_PPO_ep10000_lr0.0005_updt2048,268.545785,19.09,0.0005,2048,10,128,98.8


Donc pour avoir une meilleur visualisation sur l'ecart type et comme on peut le voir on ne passe jamais sous la barriere des 5 mais les calcules de l'ecart type dans notre csv peuvent etre remis en cause il faudras donc des test plus approfondit.

In [ ]:
def afficher_modeles_plus_stables(df, score_minimum=-150, top_n=10):
    """
    Affiche les modèles ayant l'écart-type le plus bas, 
    tout en filtrant les modèles complètement ratés grâce à un score minimum.
    """
    # 1. On filtre les modèles qui sont trop mauvais (pour éviter les IA qui crashent en boucle)
    df_filtre = df[df['eval_score_mean'] >= score_minimum].copy()
    
    if df_filtre.empty:
        print(f"🚨 Aucun modèle n'a atteint le score minimum de {score_minimum}.")
        return None
    
    # 2. On trie par écart-type CROISSANT (ascending=True) pour avoir les plus petits en premier
    top_stables = df_filtre.sort_values(by='eval_std_dev', ascending=True).head(top_n)
    
    # 3. On choisit les colonnes à afficher
    colonnes_a_voir = [
        'run_name', 
        'eval_std_dev',       # La star de cette fonction
        'eval_score_mean',    # Pour vérifier qu'il fait un bon score
        'margin_diff_pct',
        'param_learning_rate', 
        'param_update_timestep', 
        'param_ppo_epochs'
    ]
    
    # Affichage
    print(f"Voici les {len(top_stables)} modèles les plus STABLES (Score moyen >= {score_minimum}) :")
    display(top_stables[colonnes_a_voir])
    
    return top_stables

# --- Lancement de la fonction ---
# Tu peux ajuster le score_minimum. Par exemple, si tu as déjà de bons modèles, mets 100 ou 150 !
modeles_stables = afficher_modeles_plus_stables(df, score_minimum=-150, top_n=10)

🎯 Voici les 10 modèles les plus STABLES (Score moyen >= -150) :


,run_name,eval_std_dev,eval_score_mean,margin_diff_pct,param_learning_rate,param_update_timestep,param_ppo_epochs
287,test_289_PPO_ep12500_lr0.0003_updt1024,7.03,247.260704,6.240087,0.0003,1024,4
24,test_027_PPO_ep7500_lr0.0003_updt4096,8.24,42.727637,35.589429,0.0003,4096,4
215,test_215_PPO_ep10000_lr0.0003_updt4096,8.49,248.256553,8.531678,0.0003,4096,10
123,test_124_PPO_ep10000_lr0.0003_updt2048,9.03,233.402658,8.966090,0.0003,2048,4
30,test_033_PPO_ep7500_lr0.0003_updt4096,9.09,240.127755,7.912421,0.0003,4096,10
218,test_220_PPO_ep12500_lr0.0003_updt1024,9.40,250.956766,9.458483,0.0003,1024,4
120,test_121_PPO_ep10000_lr0.0003_updt2048,9.53,237.512744,8.058614,0.0003,2048,4
4,test_006_PPO_ep7500_lr0.0003_updt1024,9.63,238.287971,10.629772,0.0003,1024,4
289,test_290_PPO_ep12500_lr0.0003_updt1024,9.76,241.118348,10.294532,0.0003,1024,4
305,test_306_PPO_ep12500_lr0.0003_updt2048,10.43,245.623233,9.297659,0.0003,2048,4
